# 4 · How realistic is this market?

Realism here is a stated envelope rather than a score. Fourteen statistics
are measured against real-market bands, and what the model still gets wrong
is named rather than left for you to find. Six gaps are on the record; two
more were on it at 0.1.4 and closed at this release.


In [1]:
import pretium as pt
from pretium import envelope as env
from pretium.facts import REAL_MARKETS, band_distance

print("preset certified :", env.PRESET)
print("certified horizon:", env.CERTIFIED_HORIZON_DAYS, "trading days")

preset certified : pt-v12
certified horizon: 252 trading days


## The panel

Measured at 30 seeds, 40 instruments, 252 days.

In [2]:
print(f"{'statistic':26s} {'measured':>10s} {'band':>20s}   verdict")
for name in REAL_MARKETS:
    lo, hi = REAL_MARKETS[name]
    v = env.CERTIFIED[name]
    ok = band_distance(v, lo, hi) == 0
    print(f"{name:26s} {v:10.4f} {f'[{lo:g}, {hi:g}]':>20s}   "
          f"{'in band' if ok else 'OUT'}")

n = sum(1 for k in REAL_MARKETS
        if band_distance(env.CERTIFIED[k], *REAL_MARKETS[k]) == 0)
print(f"\n{n} of {len(REAL_MARKETS)} in band")

statistic                    measured                 band   verdict
annualised_vol_pct            32.7604             [15, 36]   in band
excess_kurtosis                6.7001            [1.6, 41]   in band
return_acf1                    0.0239        [-0.08, 0.06]   in band
abs_return_acf1                0.1107         [0.02, 0.22]   in band
abs_return_acf5                0.0428         [0.02, 0.09]   in band
abs_return_acf20               0.0040        [-0.04, 0.08]   in band
cross_sectional_corr           0.3177         [0.08, 0.56]   in band
volume_abs_return_corr         0.5599         [0.46, 0.66]   in band
leverage_effect               -0.0401           [-0.16, 0]   in band
volume_change_acf1            -0.2656        [-0.32, -0.2]   in band
corr_asymmetry                -0.0147        [-0.25, 0.45]   in band
corr_asymmetry_lagged          0.0181         [-0.2, 0.55]   in band
sector_excess_corr             0.2079         [0.11, 0.23]   in band
corr_persistence_acf1          0.1

## Nothing fails at one year, and one thing fails at two

Every statistic is inside its band at the certified horizon. That is new
in 0.2.0: the previous default missed two, and `volume_change_acf1` was
described here as unreachable.

It is still the weak one. Its two-year band is tighter than its one-year
band, and the model sits outside it there, which is why the cell below
reads 14 of 14 against the 252-day ruler and fewer against the 504-day
one. A strategy trading the change in volume is on solid ground at one
year and outside the envelope at two.

## Scoring a panel

`envelope.score` reads a panel against the ruler for its own horizon. The
same numbers score differently at 252 and 504 days, which is the mistake
this function exists to prevent.


In [3]:
panel = {k: env.CERTIFIED[k] for k in REAL_MARKETS}

near = env.score(panel, horizon_days=252)
far = env.score(panel, horizon_days=504)

print(f"252-day ruler ({near['ruler']}): {near['in_band']}/{near['of']} in band")
print(f"504-day ruler ({far['ruler']}): {far['in_band']}/{far['of']} in band")
print()
k = "excess_kurtosis"
for label, s in (("252", near), ("504", far)):
    r = s["statistics"][k]
    print(f"  {k} @{label}d: {r['measured']:.3f} vs band {r['band']}"
          f"  -> {'in' if r['in_band'] else 'OUT'}")

252-day ruler (REAL_MARKETS): 14/14 in band
504-day ruler (REAL_MARKETS_504): 12/14 in band

  excess_kurtosis @252d: 6.700 vs band (1.6, 41.0)  -> in
  excess_kurtosis @504d: 6.700 vs band (7.1, 22.0)  -> OUT


`room_sd` gives the distance inside a band in that horizon's own seed noise.
A statistic barely inside is one seed away from being outside, and a plain
band check cannot distinguish the two.

In [4]:
rows = [(k, r["room_sd"]) for k, r in near["statistics"].items()
        if r["in_band"] and r["room_sd"] is not None]
for k, room in sorted(rows, key=lambda kv: kv[1]):
    flag = "  <- thin" if room < 0.5 else ""
    print(f"  {k:26s} {room:6.2f} sd inside{flag}")

  abs_return_acf5              0.41 sd inside  <- thin
  annualised_vol_pct           0.50 sd inside
  leverage_effect              0.53 sd inside
  return_acf1                  0.68 sd inside
  abs_return_acf20             0.94 sd inside
  abs_return_acf1              0.96 sd inside
  corr_persistence_acf1        1.23 sd inside
  corr_asymmetry               1.46 sd inside
  corr_asymmetry_lagged        1.85 sd inside
  cross_sectional_corr         2.19 sd inside
  volume_abs_return_corr       2.30 sd inside
  sector_excess_corr           3.25 sd inside
  excess_kurtosis              4.35 sd inside
  volume_change_acf1           5.32 sd inside


## The gaps

Each gap names what it stops you concluding.

In [5]:
for gap in env.GAPS:
    print(f"* {gap.id}")
    print(f"    forbids: {gap.forbids}")

* horizon
    forbids: multi-year backtests, and anything keyed on volatility dynamics beyond one year
* decay-shape
    forbids: strategies whose edge depends on volatility memory beyond about lag 20 -- vol targeting and risk parity on a one-month or longer estimate
* scenario-magnitude
    forbids: sizing a scenario's impact rather than detecting it
* macro-range
    forbids: studying inflation regimes or policy crises from the endogenous economy alone
* roster-concentration
    forbids: inheriting this envelope for a sector-concentrated roster BEYOND one year -- at the certified horizon it now transfers


## Checking your own question

`check` refuses questions that fall outside the envelope, and every refusal
names the measurement behind it.

In [6]:
questions = [
    ("a one-year momentum study", dict(horizon_days=252,
                                       statistics=["return_acf1"])),
    ("a three-year study",        dict(horizon_days=756,
                                       statistics=["abs_return_acf1"])),
    ("volatility clustering decay", dict(horizon_days=252,
                                         statistics=["abs_return_acf20"])),
    ("a tech-only roster",        dict(horizon_days=252,
                                       sector_concentrated=True)),
]

for label, kwargs in questions:
    v = env.check(**kwargs)
    print(f"{label:32s} {'INSIDE' if v.inside else 'outside'}")
    if not v.inside:
        for reason in v.reasons:
            print(f"      {reason[:96]}")

a one-year momentum study        INSIDE
a three-year study               outside
      horizon 756d exceeds the certified 252d. At 504 days the model holds all 14 against horizon-matc
volatility clustering decay      outside
      abs_return_acf20 depends on the decay shape, which is a mechanism gap: log-log slope -0.953 agai
a tech-only roster               outside
      the roster is sector-concentrated, and certification was measured on a sector-balanced one. Meas


## Choosing a preset

`pt-v10` is the default and what the envelope certifies at 252 days. It has
been the default since 2026-08-26; `pt-v3` held the job before it and stays
selectable, so work published against it keeps reproducing.

`envelope.regressions` names what a panel gives up against the shipped one.
Below, `pt-v3` gives up exactly the two rows the era boundary bought: sector
co-movement and volume-change autocorrelation.

The default's own line reads 13 of 14 rather than the certified 14 because
three seeds is not thirty. `corr_persistence_acf1` reads -0.34 across these
three, against a certified thirty-seed median of +0.16 and a seed standard
deviation of 0.28: the three seeds are +0.27, -0.34 and -0.42. Read
`envelope.intervals` before trusting any one run.


In [7]:
import statistics as _stats
from pretium import facts

small = pt.Universe.random(40, seed=111)

def panel_for(preset, seeds=(1, 2, 3)):
    model = pt.ModelParams.from_preset(preset)
    panels = [facts.measure(seed=s, universe=small, days=252, model=model)
              for s in seeds]
    return {k: _stats.median(p[k] for p in panels) for k in REAL_MARKETS}

for preset in ("pt-v10", "pt-v3"):
    p = panel_for(preset)
    s = env.score(p)
    print(f"{preset}:  {s['in_band']}/{s['of']} in band at 252d"
          f"   regressions vs shipped: {env.regressions(p) or 'none'}")


pt-v10:  13/14 in band at 252d   regressions vs shipped: ['corr_persistence_acf1']


pt-v3:  12/14 in band at 252d   regressions vs shipped: ['sector_excess_corr', 'volume_change_acf1']


Three seeds is a small sample; the published figures use thirty. Treat the
counts above as indicative. What matters is the shape of the trade, and
that `regressions` reports it rather than leaving you to count by hand.

**pt-v3 for horizons at or under a year; pt-v4 for multi-year questions.**

## Summary

- Realism is a set of measurements against bands, with the failures named.
- The certified horizon is 252 days, and `check` refuses beyond it.
- Good results here do not predict real returns.

Full documentation: <https://simoncoombes.github.io/pretium/>